# Nexar event captions: Qwen2.5-VL-7B on the 451 `nexar_train` positives (L4)

Rule: `core/docs/final/KIM_RUN.md` §29, written before any caption is read. **A data artifact, not an arm**: nothing trains here, and using the captions in training needs §28 NL7 = "candidate" plus a new dated note. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **Runtime: GPU L4** (bf16, 24 GB).

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | mount, paths, deps, preflight (raw mp4s, census, split) | no | 2-3 min |
| 4 | pytest `test_nexar_caption.py` — **raises if red** | no | < 1 min |
| 5 | pin the model revision once (`revision.json`; a re-run reuses it) | no | seconds |
| 6 | smoke: 3 videos -> **read the 3 captions; stop if they are wrong in kind** | yes | 10 min (model download ~16 GB) |
| 7 | all 451 videos (resumable: a disconnect costs one video) + manifest | yes | ~45-70 min |
| 8 | QC: 30 seeded contact sheets + `qc/qc_ratings.csv` | no | 2-3 min |
| 9 | **you** fill `right` = y / n for the 30 rows on Drive, then `qc-score` (pass iff >= 27 / 30) | no | your time |

Thesis and Thesis-V2 are only read; everything new goes to `Thesis-Final/outputs/nexar_captions/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import csv
import json
import os
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data/Nexar/raw
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: the N0 census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'
os.environ['HF_HOME'] = '/content/hf'             # model weights on VM disk, not Drive FUSE

REPO = Path(f'{DRIVE_F}/kat-vad')
assert (REPO / 'core/tools/nexar_caption.py').is_file(), 'this upload has no nexar_caption -- re-upload kat-vad/'
assert '## 29. Nexar event captions' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §29 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
MODEL  = constants.NEXAR_CAPTION_MODEL
RAW    = constants.DATA_ROOT / constants.NEXAR_DATASET / 'raw'
CENSUS = Path(DRIVE_V2) / 'outputs' / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
OUT    = constants.OUTPUT_ROOT / constants.NEXAR_CAPTION_OUTPUT_DIRNAME
OUT.mkdir(parents=True, exist_ok=True)
for name, p in (('raw Nexar', RAW / constants.NEXAR_TRAIN_DIR), ('census', CENSUS)):
    assert p.exists(), f'missing {name}: {p} -- paste to Claude'
print(f'model {MODEL} | raw {RAW} | out {OUT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av
python -c "import av, transformers; print('deps OK', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6-7 need an L4'
df -h /content | tail -1

In [ ]:
import torch  # noqa: E402

from core.tools import nexar_caption as nc  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    '''Stream a child's output here; raise with its tail on failure.'''
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


census = json.loads(CENSUS.read_text())
common = ['--raw-root', RAW, '--census', CENSUS, '--out-dir', OUT]
QC = OUT / nc.QC_DIRNAME
POS = nc.train_positives(census)                               # CP1: raises on a census issue
assert len(POS) == 451, len(POS)
missing = [v for v in POS if not (RAW / census['videos'][v]['path']).is_file()]
assert not missing, f'{len(missing)} positive mp4s missing under {RAW}: {missing[:5]}'
spans = [nc.event_frame_indices(census['videos'][v])[1] for v in POS]   # CP2: raises if a span is too short
print(f'preflight OK: {len(POS)} train positives, every mp4 present | span {min(e - s for s, e in spans):.2f}-'
      f'{max(e - s for s, e in spans):.2f} s | GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

## 4. Tests on the caption tool (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_nexar_caption.py'])

## 5. Pin the model revision once (C4)

The first run asks the Hub for the current commit of the model and writes it to `revision.json`; every later run (and the resume after a disconnect) reuses it, so all 451 captions come from one revision.

In [ ]:
from huggingface_hub import HfApi  # noqa: E402

REV_FILE = OUT / 'revision.json'
if not REV_FILE.exists():
    REV_FILE.write_text(json.dumps({'model': MODEL, 'revision': HfApi().model_info(MODEL).sha}, indent=2))
REV = json.loads(REV_FILE.read_text())
assert REV['model'] == MODEL, REV
REVISION = REV['revision']
print(f'{MODEL} @ {REVISION}')

## 6. Smoke: 3 videos — **read the captions below; if they do not describe a collision, stop and paste them to Claude**

The 3 records are kept: step 7 skips them (same model, revision and prompt).

In [ ]:
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'needs a bf16 GPU (L4 / A100) -- Runtime > Change runtime type'
run([sys.executable, '-m', 'core.tools.nexar_caption', 'run', *common, '--revision', REVISION, '--limit', '3'])
for v in POS[:3]:
    r = json.loads((OUT / nc.CAPTIONS_DIRNAME / f'{v}.json').read_text())
    print(f"{v} [{r['span_s'][0]:.1f}-{r['span_s'][1]:.1f} s] {r['caption']}  forbidden={r['forbidden']}")

## 7. All 451 videos + manifest (resumable)

**A Colab disconnect is harmless**: re-run steps 1-5 and this cell; finished videos are skipped.

In [ ]:
t0 = time.time()
run([sys.executable, '-m', 'core.tools.nexar_caption', 'run', *common, '--revision', REVISION])
m = json.loads((OUT / nc.MANIFEST_FILENAME).read_text())
assert m['videos'] == len(POS) and m['revision'] == REVISION, (m['videos'], m['revision'])
print(f"{m['videos']} captions in {(time.time() - t0) / 60:.0f} min | summary {json.dumps(m['summary'])}")
drive.flush_and_unmount()
drive.mount('/content/drive')

## 8. QC sheets (§29 CP6: 30 videos, seed 2024)

Each PNG = the 8 frames the model saw, in reading order. Rate them in step 9.

In [ ]:
from IPython.display import Image, Markdown, display  # noqa: E402

run([sys.executable, '-m', 'core.tools.nexar_caption', 'qc', *common])
display(Markdown((QC / nc.QC_SHEET_FILENAME).read_text()))
for png in sorted(QC.glob('*.png')):
    print(png.stem)
    display(Image(filename=str(png), width=900))
print(f'Now open {QC / nc.QC_RATINGS_FILENAME} on Drive, fill the right column with y / n for all 30 rows, save, then run step 9.')

## 9. QC score — **after you filled `qc_ratings.csv`; paste the printed line to Claude**

In [ ]:
run([sys.executable, '-m', 'core.tools.nexar_caption', 'qc-score', '--out-dir', OUT])
print(json.loads((QC / nc.QC_SUMMARY_FILENAME).read_text()))
drive.flush_and_unmount()          # C10: every pending Drive write lands before the VM goes